# 04 - Modelagem e avaliação

**Objetivo:** treinar, comparar e escolher o modelo de recomendação (Modelagem + Avaliação).

**Lê:** `data/clean/*.parquet`  
**Gera:** modelo final em `models/` para consulta da api 

In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

proc = Path("../data/clean")
ratings = pd.read_parquet(proc/ "ratings_clean.parquet")
books = pd.read_parquet(proc/ "books_clean.parquet")
users = pd.read_parquet(proc/ "users_clean.parquet")



# 1- Filtro de esparsidade

Mantém usuários e livros com um mínimo de avaliações

In [82]:
r_counts = ratings["User-ID"].value_counts()
for k in [1, 2, 3, 5, 10, 20]:
    u = r_counts[r_counts >= k]
    pct_users_count = len(u) / len(r_counts)
    pct_ratings = u.sum() / r_counts.sum()
    print(f"min {k:>2}: mantém {pct_users_count:.1%} dos usuários e {pct_ratings:.1%} dos ratings")

min  1: mantém 100.0% dos usuários e 100.0% dos ratings
min  2: mantém 42.4% dos usuários e 89.8% dos ratings
min  3: mantém 29.7% dos usuários e 85.3% dos ratings
min  5: mantém 18.8% dos usuários e 78.7% dos ratings
min 10: mantém 9.7% dos usuários e 68.2% dos ratings
min 20: mantém 4.9% dos usuários e 56.7% dos ratings


Como a mediana de avaliações por usuário é 1, usar o segundo quartil como corte não removeria nada. Por isso foi adotado um mínimo fixo de 5 avaliações por usuário e por livro. Esse valor mantém 78.7% dos ratings mesmo removendo 82.2% dos usuários, e garante histórico mínimo para o modelo aprender preferências. A contrapartida é que o dataset filtrado representa usuários mais ativos que o usuário típico.

In [83]:
min_user, min_book = 5, 5

u = ratings["User-ID"].value_counts()
b = ratings["ISBN"].value_counts()
r = ratings[ratings["User-ID"].isin(u[u >= min_user].index) &
            ratings["ISBN"].isin(b[b >= min_book].index)].copy()

print("Antes:", ratings.shape, "| Depois:", r.shape)
print("Usuários:", r["User-ID"].nunique(), "| Livros:", r["ISBN"].nunique())

Antes: (383843, 3) | Depois: (145890, 3)
Usuários: 12293 | Livros: 13769


# 2- Treino e teste

In [84]:
train, test = train_test_split(r, test_size=0.2, random_state=42)
print(train.shape, test.shape)

(116712, 3) (29178, 3)


# 3- Baselines


In [ ]:
def avaliacao(y_true, y_pred, nome):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{nome}: RMSE {rmse:.3f} | MAE {mae:.3f}")
    return {"modelo": nome, "RMSE": rmse, "MAE": mae}

resultados = []  # acumula todos os modelos

# média global
media_global = train["Book-Rating"].mean()
resultados.append(avaliacao(test["Book-Rating"], np.full(len(test), media_global), "Média global"))

# média por livro
media_livro = train.groupby("ISBN")["Book-Rating"].mean()
pred = test["ISBN"].map(media_livro).fillna(media_global)
resultados.append(avaliacao(test["Book-Rating"], pred, "Média por livro"))

# 4- Modelos- Scikit-Learn


### 4.1- KNN item-item

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

user_id = train["User-ID"].astype("category")
book_id = train["ISBN"].astype("category")
user_map = {u: i for i, u in enumerate(user_id.cat.categories)}
book_map = {b: i for i, b in enumerate(book_id.cat.categories)}

# matriz usuário x livro
matriz = csr_matrix((train["Book-Rating"], (user_id.cat.codes, book_id.cat.codes)))
matriz_T = matriz.T.tocsr()

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(matriz_T)

Converte usuários e livros em índices, monta a matriz esparsa usuário × livro e treina o KNN sobre os livros (item-item) com similaridade de cosseno.

In [ ]:
K = 50

dist, idx = knn.kneighbors(matriz_T, n_neighbors=K + 1)
viz_idx = idx[:, 1:]       # sem o próprio livro
viz_sim = 1 - dist[:, 1:]  # similaridade

y_test = test["Book-Rating"].values

Guarda os K = 50 livros mais parecidos com cada livro, junto com a similaridade de cada um.

In [ ]:
k_suavizada = 5

def media_suavizada(col):
    s = train.groupby(col)["Book-Rating"].agg(["sum", "count"])
    return (s["sum"] + k_suavizada * media_global) / (s["count"] + k_suavizada)

media_usuario = media_suavizada("User-ID").to_dict()
mb_arr = media_suavizada("ISBN").reindex(book_id.cat.categories).values  # mesma ordem da matriz

Média de cada usuário e de cada livro puxada para a média global (k = 5), para não distorcer quem tem poucas notas. A do usuário é o fallback; a do livro é usada na correção do KNN.

In [ ]:
def prever(user, isbn, corrigir_media_livro=False):
    base = media_usuario.get(user, media_global)  # fallback
    i, j = user_map.get(user), book_map.get(isbn)
    if i is None or j is None:
        return base, True

    viz = viz_idx[j]
    notas = matriz[i, viz].toarray().ravel()
    tem_nota = notas > 0
    if not tem_nota.any():
        return base, True

    pesos = viz_sim[j][tem_nota] + 1e-9
    if corrigir_media_livro:
        desvios = notas[tem_nota] - mb_arr[viz][tem_nota]
        return float(np.clip(mb_arr[j] + np.average(desvios, weights=pesos), 1, 10)), False
    return np.average(notas[tem_nota], weights=pesos), False


def predizer(**kwargs):
    res = [prever(u, b, **kwargs) for u, b in zip(test["User-ID"], test["ISBN"])]
    pred, fb = map(np.array, zip(*res))
    return pred, fb  # previsões e máscara de fallback

`prever` usa as notas que o usuário deu aos vizinhos do livro. No KNN simples é a média ponderada pela similaridade; no KNN com médias, é a média do livro mais o desvio das notas dos vizinhos em relação às médias deles. Sem vizinhos avaliados, cai na média do usuário.

In [ ]:
pred_knn_u, fallback = predizer()
pred_media_usuario = test["User-ID"].map(media_usuario).fillna(media_global).values

resultados.append(avaliacao(y_test, pred_media_usuario, "Média por usuário"))
resultados.append(avaliacao(y_test, pred_knn_u, "KNN simples"))
print(f"Previsões no fallback: {fallback.mean():.1%}")

In [ ]:
pred_knn_i, _ = predizer(corrigir_media_livro=True)
usou = ~fallback  # só os casos com vizinhos

resultados.append(avaliacao(y_test, pred_knn_i, "KNN com médias"))
display(pd.DataFrame(resultados))

cobertos = [
    avaliacao(y_test[usou], pred_media_usuario[usou], "Média por usuário (cobertos)"),
    avaliacao(y_test[usou], pred_knn_u[usou], "KNN simples (cobertos)"),
    avaliacao(y_test[usou], pred_knn_i[usou], "KNN com médias (cobertos)"),
]
display(pd.DataFrame(cobertos))

A média por usuário mostrou-se um baseline forte (RMSE 1,60; MAE 1,24 no teste). O KNN item-based (cosseno, k = 50) com correção pela média dos livros obteve RMSE 1,62 e MAE 1,22, superando a média por usuário apenas no MAE. Nos casos em que encontrou vizinhos avaliados pelo usuário (~27% do teste), o KNN simples teve o menor MAE (1,01), mas RMSE pior (1,59 contra 1,45), e a correção pela média dos livros reduziu esse RMSE (1,53) ao custo de um MAE ligeiramente maior (1,06). Como cerca de 73% das previsões recorrem ao fallback devido à esparsidade, o ganho global dos métodos de vizinhança fica limitado.